# Examen Final - Recuperación de Información

## Sistema RAG sobre arXiv Paper Abstracts

**Autor:** David Morales

**Curso:** Recuperación de Información

**Fecha:** julio 2026


> **URL de la aplicación:** https://huggingface.co/spaces/elmo2004/rag-arxiv-abstracts

## Introducción

En este notebook se armó un sistema de Recuperación Aumentada por Generación (RAG) que responde preguntas sobre papers de arXiv, usando como corpus el dataset de Kaggle `spsayakpaul/arxiv-paper-abstracts`.

In [1]:
import sys
import os

# agregar la raíz del proyecto al path para poder importar los módulos de src/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.config import (
    GEMINI_MODEL,
    EMBEDDING_MODEL,
    CHROMA_DIR,
    COLLECTION_NAME,
    TOP_K,
    MAX_DOCS,
    DATA_DIR,
    CORPUS_CSV_PATH,
)

print("Modelo de embeddings:", EMBEDDING_MODEL)
print("Modelo de generación:", GEMINI_MODEL)
print("Carpeta de Chroma:", CHROMA_DIR)
print("Colección:", COLLECTION_NAME)
print("Máximo de documentos a indexar:", MAX_DOCS)


Modelo de embeddings: all-MiniLM-L6-v2
Modelo de generación: gemini-2.5-flash
Carpeta de Chroma: C:\Users\david\Documents\7mo semestre\ExamenFinal-RI\chroma_db
Colección: arxiv_papers
Máximo de documentos a indexar: 15000


## A. Preparación del corpus

El dataset `spsayakpaul/arxiv-paper-abstracts` tiene tres columnas: `titles`, `summaries` (el abstract) y `terms` (las categorías de arXiv, por ejemplo `cs.LG` o `cs.CV`, a veces varias por paper). En total son más de 50 mil papers.

Se decidió no usar el dataset completo por dos razones prácticas. La primera es que generar embeddings para más de 50 mil abstracts tarda bastante. La segunda es que para Hugging Face Spaces (que usa recursos limitados en el tier gratuito) un índice más chico carga más rápido y pesa menos al subirlo al repositorio. Por eso se optó por un subconjunto de 15.000 documentos.

También se eliminaron los duplicados por título, porque el dataset trae bastantes papers repetidos. Y se limpiaron los abstracts quitando saltos de línea y espacios de más, porque venían con el texto cortado en varias líneas.

In [2]:
from src.data_prep import preparar_corpus

# Descarga el dataset desde Kaggle (si no está en caché), limpia y recorta a MAX_DOCS filas.
df = preparar_corpus(max_docs=MAX_DOCS)

os.makedirs(DATA_DIR, exist_ok=True)
df.to_csv(CORPUS_CSV_PATH, index=False)

print("Shape del corpus:", df.shape)
df.head(3)


c:\Users\david\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Descargando dataset desde Kaggle (spsayakpaul/arxiv-paper-abstracts)...
Dataset descargado en: C:\Users\david\.cache\kagglehub\datasets\spsayakpaul\arxiv-paper-abstracts\versions\2
Leyendo CSV: C:\Users\david\.cache\kagglehub\datasets\spsayakpaul\arxiv-paper-abstracts\versions\2\arxiv_data_210930-054931.csv
Corpus final: 15000 documentos.
Shape del corpus: (15000, 3)


,terms,titles,summaries
0,['cs.LG'],Multi-Level Attention Pooling for Graph Neural...,Graph neural networks (GNNs) have been widely ...
1,"['cs.LG', 'cs.AI']",Decision Forests vs. Deep Networks: Conceptual...,Deep networks and decision forests (such as ra...
2,"['cs.LG', 'cs.CR', 'stat.ML']",Power up! Robust Graph Convolutional Network v...,Graph convolutional networks (GCNs) are powerf...


In [3]:
# Exploración rápida del corpus ya limpio

print("Cantidad de documentos:", len(df))
print("Columnas:", list(df.columns))
print()

# Un par de ejemplos para chequear que la limpieza funcionó bien
for i in [0, 1]:
    print("Título:", df.iloc[i]["titles"])
    print("Categorías:", df.iloc[i]["terms"])
    print("Abstract (primeros 200 caracteres):", df.iloc[i]["summaries"][:200], "...")
    print("-" * 80)

# Categorías más comunes (el campo terms puede traer varias separadas por espacio/coma
# según cómo haya quedado el string, así que se parte de forma simple)
from collections import Counter

contador_categorias = Counter()
for terms in df["terms"]:
    partes = terms.replace("[", "").replace("]", "").replace("'", "").split(",")
    for p in partes:
        p = p.strip()
        if p:
            contador_categorias[p] += 1

print("Top 10 categorías más frecuentes:")
for categoria, cantidad in contador_categorias.most_common(10):
    print(f"  {categoria}: {cantidad}")

Cantidad de documentos: 15000
Columnas: ['terms', 'titles', 'summaries']

Título: Multi-Level Attention Pooling for Graph Neural Networks: Unifying Graph Representations with Multiple Localities
Categorías: ['cs.LG']
Abstract (primeros 200 caracteres): Graph neural networks (GNNs) have been widely used to learn vector representation of graph-structured data and achieved better task performance than conventional methods. The foundation of GNNs is the ...
--------------------------------------------------------------------------------
Título: Decision Forests vs. Deep Networks: Conceptual Similarities and Empirical Differences at Small Sample Sizes
Categorías: ['cs.LG', 'cs.AI']
Abstract (primeros 200 caracteres): Deep networks and decision forests (such as random forests and gradient boosted trees) are the leading machine learning methods for structured and tabular data, respectively. Many papers have empirica ...
-------------------------------------------------------------------------

## B. Representación mediante embeddings

Para convertir los abstracts en vectores se usó `sentence-transformers` con el modelo `all-MiniLM-L6-v2`. En la práctica da un buen balance entre calidad semántica y velocidad.

In [4]:
from sentence_transformers import SentenceTransformer

modelo_embeddings = SentenceTransformer(EMBEDDING_MODEL)

# Se prueba con un abstract cualquiera del corpus
texto_ejemplo = df.iloc[0]["summaries"]
embedding_ejemplo = modelo_embeddings.encode(texto_ejemplo)

print("Texto de ejemplo (recortado):", texto_ejemplo[:150], "...")
print("Dimensión del embedding:", embedding_ejemplo.shape)
print("Primeros 10 valores:", embedding_ejemplo[:10])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1730.39it/s]


Texto de ejemplo (recortado): Graph neural networks (GNNs) have been widely used to learn vector representation of graph-structured data and achieved better task performance than c ...
Dimensión del embedding: (384,)
Primeros 10 valores: [ 0.0084572  -0.0444512   0.06236814  0.00474726  0.11086977  0.01386476
 -0.02391859 -0.04144083 -0.019924   -0.02271875]


## C. Almacenamiento y búsqueda vectorial

Para guardar los embeddings se usó ChromaDB en modo persistente (`PersistentClient`, guardando en la carpeta `chroma_db`).

También se agregó una verificación al principio: si la carpeta `chroma_db` ya existe y la colección ya tiene documentos, no se reconstruye. Solo se reconstruye si no existe o si está vacía.

In [5]:
import chromadb

cliente_chroma = chromadb.PersistentClient(path=CHROMA_DIR)

def coleccion_ya_construida(cliente, nombre_coleccion):
    """Chequea si la colección existe y ya tiene documentos cargados."""
    nombres_existentes = [c.name for c in cliente.list_collections()]
    if nombre_coleccion not in nombres_existentes:
        return False
    coleccion = cliente.get_collection(nombre_coleccion)
    return coleccion.count() > 0

if coleccion_ya_construida(cliente_chroma, COLLECTION_NAME):
    print("La colección ya existe con documentos, se carga sin reconstruir.")
    coleccion = cliente_chroma.get_collection(COLLECTION_NAME)
else:
    print("La colección no existe o está vacía, se construye desde cero (esto tarda un rato).")
    coleccion = cliente_chroma.get_or_create_collection(COLLECTION_NAME)

    ids = [str(i) for i in range(len(df))]
    # Se embebe título + abstract juntos (el título aporta bastante señal semántica),
    # pero como documento se guarda solo el abstract, que es lo que después
    # se le pasa como contexto al LLM.
    textos_a_embeber = (df["titles"] + ". " + df["summaries"]).tolist()
    documentos = df["summaries"].tolist()
    metadatos = [
        {"title": row["titles"], "terms": row["terms"]}
        for _, row in df.iterrows()
    ]

    TAMANO_LOTE = 5000  # límite máximo que acepta Chroma en un solo add()

    for inicio in range(0, len(documentos), TAMANO_LOTE):
        fin = inicio + TAMANO_LOTE

        # Se generan los embeddings del lote con el modelo de la sección B
        lote_embeddings = modelo_embeddings.encode(
            textos_a_embeber[inicio:fin], show_progress_bar=True
        ).tolist()

        coleccion.add(
            ids=ids[inicio:fin],
            embeddings=lote_embeddings,
            documents=documentos[inicio:fin],
            metadatas=metadatos[inicio:fin],
        )
        print(f"Agregado lote {inicio}-{fin} ({fin - inicio} documentos)")

print("Documentos en la colección:", coleccion.count())

La colección ya existe con documentos, se carga sin reconstruir.
Documentos en la colección: 15000


## D. Recuperación

La función `retrieve` toma la consulta del usuario, la convierte en embedding con el mismo modelo MiniLM (tiene que ser el mismo modelo que se usó para indexar, si no las distancias no significan nada) y le pide a Chroma los `top_k` documentos más cercanos. Se devuelve también la distancia de cada uno, porque se usa después para mostrarla como evidencia y para tener una idea de qué tan segura fue la búsqueda.

Chroma por defecto usa distancia coseno (según cómo se creó la colección), así que valores más chicos significan documentos más parecidos a la consulta.

In [6]:
def retrieve(query, top_k=TOP_K):
    """
    Embebe la consulta y busca los top_k documentos más parecidos en la colección.
    Devuelve una lista de dicts con: documento, titulo, categorias, distancia.
    """
    embedding_consulta = modelo_embeddings.encode(query).tolist()

    resultados = coleccion.query(
        query_embeddings=[embedding_consulta],
        n_results=top_k,
    )

    documentos_recuperados = []
    for i in range(len(resultados["ids"][0])):
        # En los metadatos de la colección las claves están en inglés
        # ("title" y "terms"), igual que en src/build_index.py.
        metadata = resultados["metadatas"][0][i]
        documentos_recuperados.append({
            "id": resultados["ids"][0][i],
            "documento": resultados["documents"][0][i],
            "titulo": metadata["title"],
            "categorias": metadata["terms"],
            "distancia": resultados["distances"][0][i],
        })
    return documentos_recuperados


# Se prueba con una de las consultas de ejemplo del enunciado
consulta_prueba = "What are the main applications of Graph Neural Networks?"
resultados_prueba = retrieve(consulta_prueba, top_k=5)

for r in resultados_prueba:
    print(f"[distancia={r['distancia']:.4f}] {r['titulo']}")
    print("  categorías:", r["categorias"])
    print("  abstract:", r["documento"][:150], "...")
    print()

[distancia=0.6894] Graph Neural Networks for Small Graph and Giant Network Representation Learning: An Overview
  categorías: ['cs.LG', 'cs.NE', 'stat.ML']
  abstract: Graph neural networks denote a group of neural network models introduced for the representation learning tasks on graph data specifically. Graph neura ...

[distancia=0.7144] Deep Learning on Graphs: A Survey
  categorías: ['cs.LG', 'cs.SI', 'stat.ML']
  abstract: Deep learning has been shown to be successful in a number of domains, ranging from acoustics, images, to natural language processing. However, applyin ...

[distancia=0.7474] Computing Graph Neural Networks: A Survey from Algorithms to Accelerators
  categorías: ['cs.LG', 'cs.DC', 'stat.ML']
  abstract: Graph Neural Networks (GNNs) have exploded onto the machine learning scene in recent years owing to their capability to model and learn from graph-str ...

[distancia=0.7602] A Fair Comparison of Graph Neural Networks for Graph Classification
  categorías: ['cs.

## E. Generación aumentada por recuperación

Para la generación se usó Gemini a través del SDK nuevo (`google-genai`), con el modelo `gemini-2.5-flash`.

El prompt se armó con reglas bastante estrictas, ya que si se lo dejaba suelto a veces completaba con conocimiento general de Gemini en vez de ceñirse a los abstracts que se le pasan. Por eso se le pidió explícitamente que:

- responda solo usando la información de los documentos de contexto que se le pasan, sin agregar nada de su propio conocimiento;
- cuando use información de un documento, cite entre corchetes cuál usó, por ejemplo `[Doc 2]`, para que se pueda verificar después contra la evidencia;
- si el contexto no alcanza para responder la pregunta, lo diga explícitamente en vez de inventar una respuesta (esto es clave para el requisito de reconocer cuando el corpus no tiene información suficiente).

La clave de la API (`GEMINI_API_KEY`) se carga desde el archivo `.env` con `python-dotenv` (eso ya lo hace `src/config.py` al importarse), así que nunca queda escrita en el código ni se sube al repositorio.

In [7]:
from google import genai

client = genai.Client()  # usa GEMINI_API_KEY del entorno (cargada por src/config.py vía dotenv)


def construir_prompt(query, documentos_recuperados):
    contexto = ""
    for i, doc in enumerate(documentos_recuperados, start=1):
        contexto += f"[Doc {i}] Título: {doc['titulo']}\n"
        contexto += f"Categorías: {doc['categorias']}\n"
        contexto += f"Abstract: {doc['documento']}\n\n"

    prompt = f"""Eres un asistente que responde preguntas sobre papers científicos de arXiv.

Reglas:
- Respondé ÚNICAMENTE usando la información de los documentos de contexto de abajo.
- Cuando uses información de un documento, citalo así: [Doc N].
- Si el contexto no tiene información suficiente para responder, decilo explícitamente en vez de inventar una respuesta.
- Respondé en español, de forma clara y concisa.

Contexto:
{contexto}

Pregunta: {query}

Respuesta:"""
    return prompt


def generate_answer(query, documentos_recuperados):
    """Arma el prompt con el contexto recuperado y le pide la respuesta a Gemini."""
    prompt = construir_prompt(query, documentos_recuperados)
    respuesta = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )
    return respuesta.text


def rag_pipeline(query, top_k=TOP_K):
    """Pipeline completo: recupera documentos y genera la respuesta con Gemini."""
    documentos_recuperados = retrieve(query, top_k=top_k)
    respuesta = generate_answer(query, documentos_recuperados)
    return {
        "query": query,
        "respuesta": respuesta,
        "documentos": documentos_recuperados,
    }


# Se prueba el pipeline completo con la misma consulta de antes
resultado_prueba = rag_pipeline("What are the main applications of Graph Neural Networks?")
print(resultado_prueba["respuesta"])

Las Redes Neuronales Gráficas (GNNs) tienen diversas aplicaciones:

*   **Clasificación de nodos y grafos** [Doc 1, Doc 4].
*   Han demostrado ser importantes en campos como **química, neurología, electrónica o redes de comunicación** [Doc 3].
*   Se utilizan para **representaciones relacionales** y para **modelar dominios de datos irregulares** como **nubes de puntos y grafos sociales** [Doc 5].


## F. Presentación de evidencias

El enunciado pide que se puedan verificar las evidencias detrás de cada respuesta, así que se armó una función que imprime, para una consulta dada, la respuesta generada y abajo la lista de documentos usados como contexto (con título, categorías, distancia y un fragmento del abstract). La idea es que cualquiera pueda mirar la respuesta y al lado los `[Doc N]` citados y chequear a mano si tiene sentido lo que dijo el modelo.

In [8]:
def mostrar_resultado_con_evidencias(resultado):
    """Imprime la respuesta generada junto con las evidencias (documentos) usadas."""
    print("=" * 90)
    print("CONSULTA:", resultado["query"])
    print("=" * 90)
    print()
    print("RESPUESTA GENERADA:")
    print(resultado["respuesta"])
    print()
    print("-" * 90)
    print("EVIDENCIAS UTILIZADAS:")
    print("-" * 90)
    for i, doc in enumerate(resultado["documentos"], start=1):
        print(f"[Doc {i}] {doc['titulo']}")
        print(f"        categorías: {doc['categorias']}")
        print(f"        distancia: {doc['distancia']:.4f}")
        print(f"        fragmento: {doc['documento'][:200]}...")
        print()


# Se usa el resultado de la sección E como ejemplo
mostrar_resultado_con_evidencias(resultado_prueba)

CONSULTA: What are the main applications of Graph Neural Networks?

RESPUESTA GENERADA:
Las Redes Neuronales Gráficas (GNNs) tienen diversas aplicaciones:

*   **Clasificación de nodos y grafos** [Doc 1, Doc 4].
*   Han demostrado ser importantes en campos como **química, neurología, electrónica o redes de comunicación** [Doc 3].
*   Se utilizan para **representaciones relacionales** y para **modelar dominios de datos irregulares** como **nubes de puntos y grafos sociales** [Doc 5].

------------------------------------------------------------------------------------------
EVIDENCIAS UTILIZADAS:
------------------------------------------------------------------------------------------
[Doc 1] Graph Neural Networks for Small Graph and Giant Network Representation Learning: An Overview
        categorías: ['cs.LG', 'cs.NE', 'stat.ML']
        distancia: 0.6894
        fragmento: Graph neural networks denote a group of neural network models introduced for the representation learning tasks

## G. Interfaz web conversacional

Para la interfaz se usó Gradio porque permite armar algo funcional en pocas líneas y se integra directo con Hugging Face Spaces. La interfaz es bastante simple: una caja de texto para escribir la consulta en lenguaje natural, un botón para enviarla, y dos zonas de salida: una con la respuesta generada y otra con las evidencias (los documentos recuperados, con su título, categorías, distancia y fragmento del abstract).

## H. Despliegue en la nube

Para el despliegue se eligió Hugging Face Spaces con el SDK de Gradio, que se integra directo con el `Blocks` de la sección G y no obliga a configurar servidores ni contenedores a mano.

La clave `GEMINI_API_KEY` no se subió como archivo `.env` al repositorio (eso expondría la clave), sino que se configuró como *secret* en la configuración del Space, y el código la lee de la variable de entorno igual que en local.

La app quedó desplegada y funcionando en https://huggingface.co/spaces/elmo2004/rag-arxiv-abstracts. La celda de abajo hace un `requests.get` a esa URL para confirmar que el Space responde (status 200).

In [10]:
import requests

URL_DEPLOY = "https://huggingface.co/spaces/elmo2004/rag-arxiv-abstracts" 

try:
    respuesta_http = requests.get(URL_DEPLOY, timeout=10)
    print("Status code:", respuesta_http.status_code)
    if respuesta_http.status_code == 200:
        print("El Space respondió correctamente.")
    else:
        print("El Space respondió pero con un status distinto de 200.")
except Exception as error:
    print("No se pudo contactar la URL  :", error)


Status code: 200
El Space respondió correctamente.


## I. Evaluación del sistema y de la generación

Se usaron las 4 consultas de ejemplo del enunciado, más dos adicionales: una consulta claramente fuera del dominio del corpus ("What is the best recipe for chocolate cake?"), para ver si el sistema reconoce que no tiene información y no inventa una respuesta con papers que no tienen nada que ver; y una consulta que obliga a comparar/integrar información de subáreas distintas (visión y NLP), para poder evaluar mejor la capacidad de integración de varios documentos.

Los juicios de las celdas de abajo corresponden a las salidas concretas que quedan impresas en la celda anterior.

In [10]:
consultas_evaluacion = [
    "What are the main applications of Graph Neural Networks?",
    "How is reinforcement learning used in robotics?",
    "Recent advances in diffusion models for image generation.",
    "Techniques for improving retrieval-augmented generation systems.",
    "What is the best recipe for chocolate cake?",
    "Compare the use of attention mechanisms in both computer vision and natural language processing.",
]

resultados_evaluacion = {}

for consulta in consultas_evaluacion:
    resultados_evaluacion[consulta] = rag_pipeline(consulta, top_k=5)
    print("Procesada:", consulta)


Procesada: What are the main applications of Graph Neural Networks?
Procesada: How is reinforcement learning used in robotics?
Procesada: Recent advances in diffusion models for image generation.
Procesada: Techniques for improving retrieval-augmented generation systems.
Procesada: What is the best recipe for chocolate cake?
Procesada: Compare the use of attention mechanisms in both computer vision and natural language processing.


In [11]:
# Se muestra cada resultado con sus evidencias, para poder analizarlos abajo
for consulta in consultas_evaluacion:
    mostrar_resultado_con_evidencias(resultados_evaluacion[consulta])
    print()

CONSULTA: What are the main applications of Graph Neural Networks?

RESPUESTA GENERADA:
Las Redes Neuronales Gráficas (GNNs) tienen diversas aplicaciones, incluyendo:

*   Tareas de aprendizaje de representación en datos gráficos [Doc 1, Doc 5].
*   Tareas de clasificación de nodos y grafos [Doc 1, Doc 4].
*   Campos como la química, neurología, electrónica y redes de comunicación [Doc 3].
*   Modelado de dominios de datos irregulares como nubes de puntos y grafos sociales [Doc 5].

------------------------------------------------------------------------------------------
EVIDENCIAS UTILIZADAS:
------------------------------------------------------------------------------------------
[Doc 1] Graph Neural Networks for Small Graph and Giant Network Representation Learning: An Overview
        categorías: ['cs.LG', 'cs.NE', 'stat.ML']
        distancia: 0.6894
        fragmento: Graph neural networks denote a group of neural network models introduced for the representation learning tasks 

### Consulta 1: "What are the main applications of Graph Neural Networks?"

**Corrección:** buena. Listó aplicaciones como el aprendizaje de representación sobre grafos, la clasificación de nodos y grafos, y campos donde los datos son relacionales (química, neurología, electrónica, redes de comunicación). Todo eso aparecía efectivamente en los abstracts recuperados y no se detectó nada inventado.

**Relevancia:** alta. Las cinco distancias quedaron bastante bajas (entre 0.69 y 0.78) y los cinco documentos eran surveys de GNNs.

**Fidelidad a las evidencias:** buena. Citó `[Doc 1]` para el aprendizaje de representación, `[Doc 3]` para los campos de aplicación y `[Doc 4]` (que justamente es "A Fair Comparison of GNNs for Graph Classification") para la clasificación, y cada afirmación se corresponde con su abstract. El único documento que no usó fue el `[Doc 2]`, que igual era pertinente; no se lo tomó como error.

**Integración de varios documentos:** buena, combinó información de 4 de los 5 papers en vez de quedarse con uno solo.

**Reconocimiento de información insuficiente:** no aplica, había información de sobra.

Valoración general: **buena**, aunque un poco genérica.

### Consulta 2: "How is reinforcement learning used in robotics?"

**Corrección:** correcta. Habló de aprendizaje autónomo para reducir el conocimiento de ingeniería en control, del uso de procesos gaussianos para aprender de forma más eficiente en robótica (`[Doc 1]`), del descubrimiento de relaciones causales para mejorar la eficiencia de datos (`[Doc 3]`) y de implementar políticas evitando el ruido blanco que provoca movimientos bruscos en los robots (`[Doc 4]`). Todo eso está en los abstracts.

**Relevancia:** buena, respondió directo la pregunta.

**Fidelidad a las evidencias:** en general fiel, pero hay un detalle que llama la atención: para el `[Doc 2]` ("Temporal Aware Deep Reinforcement Learning") el modelo dijo que "tiene implicaciones en robótica", cuando ese abstract habla de DRL basado en imágenes en general.

**Integración de varios documentos:** buena. Un punto a favor: el `[Doc 5]` era "Reinforcement Learning in R", que en realidad es sobre un paquete de R y no tiene nada que ver con robótica, y el modelo directamente no lo usó.

**Reconocimiento de información insuficiente:** no aplica, había información suficiente.

Valoración general: **aceptable/buena**. La respuesta es correcta y bien armada.

### Consulta 3: "Recent advances in diffusion models for image generation."

Con "diffusion models for image generation" se hacía referencia a los modelos generativos de difusión (los tipo Stable Diffusion / DDPM), pero el sistema recuperó otra cosa completamente distinta: el `[Doc 1]` que citó es "Manifold-aware Synthesis of High-resolution Diffusion from Structural Imaging", que trata sobre *diffusion imaging* de resonancia magnética del cerebro (tensores de difusión, dODFs). Es decir, la palabra "diffusion" es ambigua y el corpus no tiene papers de modelos generativos de difusión.

**Corrección:** mala respecto de lo que la pregunta realmente quería.

**Relevancia:** baja. Se nota además en las distancias, que fueron bastante más altas que en las consultas 1 y 2 (el `[Doc 1]` a 0.8751 y el resto ya por encima de 0.97), señal de que ningún documento era un buen match.

**Fidelidad a las evidencias:** buena, no alucinó; el problema no fue la generación sino la recuperación.

**Integración de varios documentos:** mala, usó un solo documento. Aunque en este caso ignorar los otros cuatro (que eran de segmentación) estuvo bien.

**Reconocimiento de información insuficiente:** dado que ningún documento hablaba de modelos generativos de difusión, lo ideal habría sido que el sistema avisara que el corpus no cubre ese tema, en vez de responder con seguridad sobre el paper de resonancia.

Valoración general: **mala**. Es un buen ejemplo de cómo un problema de recuperación arrastra a toda la respuesta.

### Consulta 4: "Techniques for improving retrieval-augmented generation systems."

**Corrección:** floja. La respuesta describe dos trabajos: el HRGR-Agent (`[Doc 1]`), que es un agente de recuperación-generación reforzado pero para *generación de reportes de imágenes médicas*, y el marco Retrieve-and-Edit (`[Doc 3]`), que recupera un ejemplo y lo edita para predecir salidas estructuradas. Son ideas emparentadas con recuperar-y-generar, pero ninguno es realmente sobre "mejorar sistemas RAG" en el sentido moderno (RAG sobre LLMs).

**Relevancia:** aceptable tirando a floja. Se ve en las distancias, que fueron las más altas de todas las consultas dentro de dominio: los cinco documentos por encima de 1.17. Ese número ya es un aviso de que el corpus no cubre bien RAG.

**Fidelidad a las evidencias:** buena en el sentido literal: lo que dice de cada documento se corresponde con su abstract. No inventó técnicas que no estuvieran ahí.

**Integración de varios documentos:** aceptable, combinó dos papers (`[Doc 1]` y `[Doc 3]`) e ignoró los otros tres, que eran aún menos relevantes.

**Reconocimiento de información insuficiente:** malo. Con distancias tan altas, lo ideal habría sido que dijera que el corpus tiene poca cobertura de este tema.

Valoración general: **aceptable**. Funciona y es fiel a lo que recuperó, pero deja ver la misma limitación del caso anterior.

### Consulta 5 (fuera de dominio): "What is the best recipe for chocolate cake?"

Esta fue la consulta trampa, para ver qué hace el sistema cuando le preguntan algo que no tiene nada que ver con el corpus. Y respondió justo como se esperaba: *"El contexto proporcionado no contiene información sobre la mejor receta de pastel de chocolate."*

**Corrección:** no aplica una respuesta "correcta" sobre el tema, lo importante era el comportamiento, y fue el adecuado.

**Relevancia:** buena, reconoció explícitamente que la pregunta cae fuera del dominio.

**Fidelidad a las evidencias:** buena, no citó ningún documento como respaldo de una respuesta real, que es exactamente lo que corresponde cuando no hay evidencia útil. Los documentos que recuperó (sobre GANs, arte generativo, panoramas) tenían distancias de 1.5 en adelante, las más altas de toda la evaluación.

**Integración de varios documentos:** no aplica.

**Reconocimiento de información insuficiente:** este fue el punto fuerte de esta consulta. El prompt de la sección E funcionó y el modelo no alucinó una receta.

Valoración general: **buena**.

### Consulta 6 (integración de varios documentos): "Compare the use of attention mechanisms in both computer vision and natural language processing."

**Corrección:** correcta. Explicó que los mecanismos de atención tuvieron éxito primero en NLP con los transformers y que su extensión a visión dio lugar a los vision transformers (ViTs), señalando también que los modelos puramente de atención requieren muchos datos y cómputo (`[Doc 1]`). Después sumó el uso de atención para fusionar visión y lenguaje en VQA (`[Doc 3]`) y en tareas de visión-a-lenguaje con atención guiada por texto y por semántica (`[Doc 4]`).

**Relevancia:** buena, respondió a la comparación pedida sin quedarse en un solo lado.

**Fidelidad a las evidencias:** buena, las afirmaciones se corresponden con los abstracts citados.

**Integración de varios documentos:** buena en cantidad (integró `[Doc 1]`, `[Doc 3]` y `[Doc 4]`), pero con un matiz que vale la pena señalar: los cinco documentos recuperados son de visión o de visión-lenguaje (todos con `cs.CV`), no hay ninguno puramente de NLP.

**Reconocimiento de información insuficiente:** no aplica, había información de ambos lados (aunque desbalanceada).

Valoración general: **buena**. Fue la consulta que mejor mostró la capacidad de integrar varios documentos, con la salvedad de que el desbalance del corpus se nota en que el lado de NLP quedó más flojo.

### Tabla resumen de la evaluación

| Consulta | Corrección | Relevancia | Fidelidad a evidencias | Integración de varios docs | Reconoce info insuficiente |
|---|---|---|---|---|---|
| 1. Graph Neural Networks | Buena | Buena | Buena | Buena | No aplica |
| 2. RL en robótica | Buena | Buena | Aceptable | Buena | No aplica |
| 3. Diffusion models | Mala | Mala | Buena | Mala | Mala |
| 4. Mejoras a RAG | Aceptable | Aceptable | Buena | Aceptable | Mala |
| 5. Receta de torta (fuera de dominio) | No aplica | Buena | Buena | No aplica | Buena |
| 6. Atención en visión y NLP | Buena | Buena | Buena | Buena | No aplica |